# Ana Deney — 5 Flower Types — 90 Koşu

Bu notebook **flowers5** için dondurulmuş ana CNN deneyini çalıştırır: **6 aktivasyon × 5 optimizer × 3 bağımsız seed = 90 koşu**.

Ana protokol: 50 epoch, batch size 128, seedler 101/202/303, NVIDIA T4, scheduler yok, early stopping yok. Learning-rate değerleri tamamlanmış pilotlardan önce sabitlenmiştir. Test seti yalnız her eğitim tamamen bittikten sonra final değerlendirmede kullanılır.

Kaynak: Kaggle 5 Flower Types Classification Dataset (kausthubkannan/5-flower-types-classification-dataset). Sabit stratified %80/%10/%10 train-validation-test ayrımı kullanılır.

**Veri kaynağı internetten/KaggleHub'dan indirilmeyecek.** Drive'daki Flowers ZIP kullanılacak. Beş sınıfın `Lilly, Lotus, Orchid, Sunflower, Tulip` olması ve her sınıfta 1.000 görüntü bulunması doğrulanmadan 90 koşu başlamayacak.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)


## Güncel GitHub sürümünü al


In [ ]:
import io, os, shutil, tarfile, requests, json
from pathlib import Path

OWNER='erdemci07'; REPO='derin-ogrenme-egitim-dinamikleri'; CODE_COMMIT='1314d438539eecdec812d9ae266d325e28b881b1'
repo_dir=Path('/content/derin-ogrenme-egitim-dinamikleri')
extract_root=Path('/content/_tez_repo_extract')
shutil.rmtree(repo_dir, ignore_errors=True)
shutil.rmtree(extract_root, ignore_errors=True)
extract_root.mkdir(parents=True, exist_ok=True)
r=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/{CODE_COMMIT}', timeout=60)
r.raise_for_status(); commit_sha=r.json()['sha']
if commit_sha != CODE_COMMIT:
    raise RuntimeError(f'Beklenen kod commit uyuşmuyor: {commit_sha} != {CODE_COMMIT}')
a=requests.get(f'https://api.github.com/repos/{OWNER}/{REPO}/tarball/{commit_sha}', timeout=120)
a.raise_for_status()
with tarfile.open(fileobj=io.BytesIO(a.content), mode='r:gz') as tar:
    tar.extractall(extract_root)
kokler=[p for p in extract_root.iterdir() if p.is_dir()]
if len(kokler)!=1:
    raise RuntimeError('GitHub arşiv yapısı beklenenden farklı.')
shutil.move(str(kokler[0]), str(repo_dir)); os.chdir(repo_dir)
os.environ['TEZ_GIT_COMMIT']=commit_sha
print('Repo hazır:', commit_sha)


## Google Drive ve kalıcı yollar


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_KOK=Path('/content/drive/MyDrive')
DRIVE_VERI=DRIVE_KOK/'tez_datasets'
DRIVE_PROJE=DRIVE_KOK/'tez_deney_sonuclari'
DRIVE_AYAR=DRIVE_PROJE/'ortak_ayarlar'
DRIVE_CIKTI=DRIVE_PROJE/'ana_deney'/'flowers5'
DRIVE_AYAR.mkdir(parents=True, exist_ok=True)
DRIVE_CIKTI.mkdir(parents=True, exist_ok=True)
DRIVE_VERI.mkdir(parents=True, exist_ok=True)
os.environ['TEZ_VERI_KLASORU']=str(DRIVE_VERI)
os.environ['TEZ_SPLIT_DOSYASI']=str(DRIVE_AYAR/'ana_flowers5_split.json')
os.environ['TEZ_STATS_DOSYASI']=str(DRIVE_AYAR/'ana_flowers5_stats.json')
os.environ['TEZ_CIKTI_KLASORU']=str(DRIVE_CIKTI)
print('Sonuç klasörü:', DRIVE_CIKTI)


## Bağımlılıklar


In [ ]:
subprocess.run(['python','-m','pip','install','-q','-r','requirements_colab.txt'], check=True)
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'
print('GPU:', gpu)
if not torch.cuda.is_available() or 'T4' not in gpu.upper():
    raise RuntimeError(f'Ana deney için NVIDIA T4 seçilmelidir. Mevcut: {gpu}')


## Veri setini hazırla / doğrula


In [ ]:
adaylar=[
    DRIVE_VERI/'flowers5.zip',
    DRIVE_VERI/'5-flower-types-classification-dataset.zip',
]
bulunan=[p for p in adaylar if p.is_file()]
if len(bulunan) != 1:
    raise FileNotFoundError(
        'Flowers-5 Drive ZIP henüz tek ve doğrulanabilir biçimde bulunamadı. '
        f'Beklenen dosyalardan biri: {[str(p) for p in adaylar]} | bulunan={bulunan}'
    )
print('Flowers-5 Drive ZIP bulundu:', bulunan[0])
print('Ön kontrolde ZIP açılabilirliği ve Lilly/Lotus/Orchid/Sunflower/Tulip sınıflarında 1000’er görüntü doğrulanacak.')


## Ana deney ön kontrolü

Bu hücre 90 eğitim başlamadan önce split, normalizasyon, giriş boyutu, altı aktivasyon ve beş optimizer zincirini gerçek bir batch üzerinde doğrular. **ÖN KONTROL BAŞARILI** yazmadan ana koşulara geçmez.


In [ ]:
import importlib
import calistir_ana_deney as ana
importlib.reload(ana)
ana.ana(['--veri-seti','flowers5','--sadece-on-kontrol'])


## 90 ana koşuyu çalıştır / devam ettir

Runtime kesilirse aynı notebook yeniden **Run all** yapılabilir. Tamamlanan koşular atlanır; yarım koşu checkpoint'ten sürdürülür.


In [ ]:
importlib.reload(ana)
ana.ana(['--veri-seti','flowers5'])


## İlerleme özeti


In [ ]:
tamam={}; kararsiz={}
for p in DRIVE_CIKTI.glob('*/tamamlandi.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8')); tamam[d['run_id']]=d
    except Exception:
        pass
for p in DRIVE_CIKTI.glob('*/kararsiz.json'):
    try:
        d=json.loads(p.read_text(encoding='utf-8'))
        if d['run_id'] not in tamam: kararsiz[d['run_id']]=d
    except Exception:
        pass
deger=len(tamam)+len(kararsiz)
print(f'Değerlendirilen: {deger}/90 | tamam={len(tamam)} | kararsız={len(kararsiz)}')
if deger < 90:
    print('Kesinti olursa aynı notebooku yeniden Run all yapın.')
else:
    print('Bu veri setinin 90 ana koşusu tamamlandı/değerlendirildi.')
